# Task 3 — Query Optimization: Broadcast Join

## Overview

This notebook tests broadcast join using Query 1 (monthly_zone_demand), which joins taxi_trips (several million rows) against taxi_zones (265 rows). Broadcast join copies the small table to every executor once, so the large table never gets shuffled across the network. The alternative, a shuffle join, redistributes both tables by join key before matching them.

taxi_zones is well under Spark's default auto broadcast threshold (10 MB), so Spark already picks broadcast join for this query with no code change. Because of this, the comparison here runs the other way around: the query exactly as written is the optimized side, and the baseline adds a hint that forces Spark away from broadcast, so we can measure what the alternative would cost.

## Query design

Both versions are Query 1 exactly as written in the Task 1/2 library:

- Baseline: adds a /*+ SHUFFLE_HASH(z) */ hint, forcing Spark to shuffle both tables and build a hash table per partition instead of broadcasting taxi_zones.
- Optimized: no hint. This is Spark's default plan, which already broadcasts taxi_zones.

## Methodology

Each version runs one discarded warm-up query, then five timed repetitions in a fresh kernel. We report the median. This notebook uses its own SparkSession, separate from the other three.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))

from data_intensive_computing.benchmark_helpers import (
    build_spark_session,
    register_delta_tables,
    timed_run,
    capture_explain,
    verify_same_results,
    append_benchmark_record,
    find_project_root,
)

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "artifacts" / "task3_benchmark_results.jsonl"

spark = build_spark_session(app_name="Task3-Broadcast-Join")
register_delta_tables(spark, PROCESSED_DIR)

:: loading settings :: url = jar:file:/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rexchang/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rexchang/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fd6c5c75-ec8c-4f0b-a5dd-03a7d26e836a;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in c

Registered taxi_trips: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/taxi_trips
Registered weather: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/weather
Registered air_quality: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/air_quality
Registered taxi_zones: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/taxi_zones
Registered integrated_taxi_trips: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/integrated_taxi_trips


26/09/17 21:11:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Query pair

In [ ]:
# Baseline: the SHUFFLE_HASH hint forces Spark to redistribute BOTH tables by
# join key (pickup_location_id / location_id) across partitions and build a
# hash table per partition, instead of copying taxi_zones to every executor.
BASELINE_SQL = """
SELECT /*+ SHUFFLE_HASH(z) */
    t.pickup_year,
    t.pickup_month,
    t.pickup_location_id,
    COALESCE(z.borough, 'Unknown') AS pickup_borough,
    COALESCE(z.zone, 'Unknown') AS pickup_zone,
    COUNT(t.trip_id) AS trip_count
FROM taxi_trips AS t
LEFT JOIN taxi_zones AS z
    ON t.pickup_location_id = z.location_id
GROUP BY
    t.pickup_year,
    t.pickup_month,
    t.pickup_location_id,
    COALESCE(z.borough, 'Unknown'),
    COALESCE(z.zone, 'Unknown')
ORDER BY t.pickup_year, t.pickup_month, trip_count DESC, t.pickup_location_id
"""

# Optimized: Query 1 exactly as written in the Task 1/2 library. No hint is
# needed — taxi_zones (265 rows) is far below the default 10 MB
# auto-broadcast threshold, so Spark already broadcasts it by default.
OPTIMIZED_SQL = """
SELECT
    t.pickup_year,
    t.pickup_month,
    t.pickup_location_id,
    COALESCE(z.borough, 'Unknown') AS pickup_borough,
    COALESCE(z.zone, 'Unknown') AS pickup_zone,
    COUNT(t.trip_id) AS trip_count
FROM taxi_trips AS t
LEFT JOIN taxi_zones AS z
    ON t.pickup_location_id = z.location_id
GROUP BY
    t.pickup_year,
    t.pickup_month,
    t.pickup_location_id,
    COALESCE(z.borough, 'Unknown'),
    COALESCE(z.zone, 'Unknown')
ORDER BY t.pickup_year, t.pickup_month, trip_count DESC, t.pickup_location_id
"""

## Execution time

In [3]:
baseline_timing = timed_run(spark, "monthly_zone_demand_shuffle_hash", BASELINE_SQL, repetitions=5)
optimized_timing = timed_run(spark, "monthly_zone_demand_broadcast", OPTIMIZED_SQL, repetitions=5)

print("Baseline (SHUFFLE_HASH hint):", baseline_timing)
print("Optimized (default — broadcast):", optimized_timing)

Baseline (SHUFFLE_HASH hint): {'query_name': 'monthly_zone_demand_shuffle_hash', 'row_count': 793, 'repetitions': 5, 'all_times_seconds': [1.3848, 1.2322, 1.2314, 1.2216, 1.1296], 'median_seconds': 1.2314, 'min_seconds': 1.1296, 'max_seconds': 1.3848}
Optimized (default — broadcast): {'query_name': 'monthly_zone_demand_broadcast', 'row_count': 793, 'repetitions': 5, 'all_times_seconds': [0.4411, 0.362, 0.3395, 0.3449, 0.3334], 'median_seconds': 0.3449, 'min_seconds': 0.3334, 'max_seconds': 0.4411}


## Physical plan comparison

The field to check is whether `taxi_trips` itself is shuffled before the
join. In the broadcast plan, only `taxi_zones` is redistributed
(`BroadcastExchange`, copied to every executor); `taxi_trips` is read and
joined locally with no `Exchange` step of its own before the join. In the
shuffle-hash plan, **both** sides are shuffled (`Exchange` with
`hashpartitioning` on the join key) — including `taxi_trips`, which is the
cost broadcast join is designed to avoid.

In [4]:
baseline_plan = capture_explain(spark.sql(BASELINE_SQL), mode="formatted")
optimized_plan = capture_explain(spark.sql(OPTIMIZED_SQL), mode="formatted")

print("=" * 30, "BASELINE PLAN (SHUFFLE_HASH)", "=" * 30)
print(baseline_plan)

============================== BASELINE PLAN (SHUFFLE_HASH) ==============================
== Physical Plan ==
AdaptiveSparkPlan (12)
+- Sort (11)
   +- Exchange (10)
      +- HashAggregate (9)
         +- HashAggregate (8)
            +- Project (7)
               +- ShuffledHashJoin LeftOuter BuildRight (6)
                  :- Exchange (2)
                  :  +- Scan parquet  (1)
                  +- Exchange (5)
                     +- Filter (4)
                        +- Scan parquet  (3)


(1) Scan parquet 
Output [4]: [pickup_location_id#7, trip_id#23, pickup_year#27, pickup_month#28]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/taxi_trips]
ReadSchema: struct<pickup_location_id:int,trip_id:string>

(2) Exchange
Input [4]: [pickup_location_id#7, trip_id#23, pickup_year#27, pickup_month#28]
Arguments: hashpartitioning(pickup_location_id#7, 8), ENSURE

In [5]:
print("=" * 30, "OPTIMIZED PLAN (BROADCAST)", "=" * 30)
print(optimized_plan)

============================== OPTIMIZED PLAN (BROADCAST) ==============================
== Physical Plan ==
AdaptiveSparkPlan (12)
+- Sort (11)
   +- Exchange (10)
      +- HashAggregate (9)
         +- Exchange (8)
            +- HashAggregate (7)
               +- Project (6)
                  +- BroadcastHashJoin LeftOuter BuildRight (5)
                     :- Scan parquet  (1)
                     +- BroadcastExchange (4)
                        +- Filter (3)
                           +- Scan parquet  (2)


(1) Scan parquet 
Output [4]: [pickup_location_id#7, trip_id#23, pickup_year#27, pickup_month#28]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/taxi_trips]
ReadSchema: struct<pickup_location_id:int,trip_id:string>

(2) Scan parquet 
Output [3]: [location_id#90, borough#91, zone#92]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/rexchan

## Correctness verification

We check both queries return the same rows using a bidirectional set difference. `rows_only_in_before` and `rows_only_in_after` must both be zero for the results to count as identical. Forcing a different join strategy should only change how the result is computed, not the result itself.

In [6]:
correctness = verify_same_results(spark, BASELINE_SQL, OPTIMIZED_SQL)
print(correctness)
assert correctness["correctness_verified"], "Baseline and optimized queries disagree — investigate before recording a result."


{'correctness_verified': True, 'rows_only_in_before': 0, 'rows_only_in_after': 0}


## Benchmark record

In [7]:
speedup = round(baseline_timing["median_seconds"] / optimized_timing["median_seconds"], 2)

record = {
    "query_name": "monthly_zone_demand",
    "technique": "broadcast_join",
    "before_time_sec": baseline_timing["median_seconds"],
    "after_time_sec": optimized_timing["median_seconds"],
    "speedup": speedup,
    "plan_change_summary": (
        "Forcing a shuffle-hash join (SHUFFLE_HASH hint) requires an Exchange "
        "(hash repartitioning by join key) on both taxi_trips and taxi_zones before "
        "the join. The default plan (no hint) instead broadcasts taxi_zones via a "
        "BroadcastExchange and avoids shuffling taxi_trips entirely, since taxi_zones "
        "(265 rows) is well under the default auto-broadcast threshold."
    ),
    "correctness_verified": correctness["correctness_verified"],
}

append_benchmark_record(record, RESULTS_PATH)
print(f"Speedup: {speedup}x — appended to {RESULTS_PATH}")

Speedup: 3.57x — appended to /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/artifacts/task3_benchmark_results.jsonl


## Discussion

taxi_zones has a fixed 265 rows regardless of how much trip data exists, while taxi_trips keeps growing every month. That size gap is why this join is a natural fit for broadcast join.

Spark already picks broadcast join for this query by default, since taxi_zones is well under the 10 MB auto broadcast threshold. So this experiment did not require any change to the production query. Its value is confirming that Spark's default choice is correct, and measuring what it would cost if that stopped being true.

If the platform expands to ten cities, each city's own zone table should still be small. The real risk is if all ten cities' zone lookups get merged into one combined table. If that combined table crosses the broadcast threshold, Spark would silently fall back to a shuffle join, with no warning that anything changed.

Broadcasting copies the entire small table to every executor's memory. That cost is negligible here, but it's the general tradeoff broadcast join makes, worth stating even though it doesn't matter at this table size.

Result: 3.57x speedup. The baseline plan shows both tables being shuffled, including the multi-million-row taxi_trips table. The default plan only shuffles the small table.

In [8]:
spark.stop()